Виконали студенти групи КІ-51 мп
* Архипов Яків,
* Польнікова Поліна,
* Ткаченко Марія

# **Імпорт бібліотек**

In [ ]:
!pip install -q transformers encodec datasets pytorch-lightning soundfile librosa evaluate speechbrain jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 28.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torchaudio
import pytorch_lightning as pl
import torch.nn.functional as F
import pytorch_lightning as pl
import os
import pandas as pd
import evaluate
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import pipeline
from speechbrain.inference.speaker import EncoderClassifier
from pytorch_lightning.callbacks import ModelCheckpoint
from IPython.display import display, Audio
from transformers import GPT2LMHeadModel, GPT2Config, AutoTokenizer
from encodec import EncodecModel
from encodec.utils import convert_audio
from datasets import load_dataset, Audio
from torch.utils.data import DataLoader, Subset, random_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Архітектура моделі (GPT-2 для TTS)**

У даному фрагменті коду реалізовано клас `GPT2TTS`, який адаптує стандартну архітектуру авторегресивного трансформера (GPT-2) для вирішення мультимодальної задачі генерації мовлення (Text-to-Speech). Основна ідея полягає у зведенні задачі TTS до класичного мовного моделювання (Language Modeling), де аудіо розглядається як послідовність дискретних токенів, згенерованих кодеком (наприклад, Encodec).

**1. Ініціалізація та об'єднання словників (`__init__`)**
Для того, щоб одна модель могла одночасно обробляти текст та звук, створюється єдиний розширений словник (`total_vocab_size`). Він містить:
*   Словник текстового токенізатора.
*   Словник аудіокодека.
*   Чотири спеціальні токени (`BOS_TEXT`, `EOS_TEXT`, `BOS_AUDIO`, `EOS_AUDIO`), які виконують роль структурних маркерів. Вони дозволяють трансформеру чітко розрізняти модальності та розуміти, коли закінчується текстова умова і починається цільова генерація аудіо.

Сама конфігурація `GPT2Config` була оптимізована: кількість шарів (`n_layer`) зменшено до 6, а розмірність ембедингів (`n_embd`) — до 512. Це дозволяє суттєво пришвидшити експерименти та зменшити вимоги до відеопам'яті без повної втрати репрезентативної здатності моделі.

**2. Прямий прохід для навчання (`forward`)**
Метод `forward` є максимально мінімалістичним і делегує обчислення базовій моделі GPT-2. При передачі аргументу `labels` (який в авторегресивних моделях є зміщеною на один крок послідовністю `input_ids`), модель "під капотом" автоматично розраховує функцію втрат `CrossEntropyLoss`. Механізм маскування (передача значення `-100` для текстової частини лейблів) реалізується на рівні DataModule, що змушує модель вчитися прогнозувати виключно аудіо-токени.

**3. Процес інференсу та семплювання (`generate`)**
Метод генерації реалізує авторегресивний підхід до створення аудіо:
*   **Формування промпту:** На вхід моделі подається структурна послідовність: маркер початку тексту + текстові токени + маркер кінця тексту + маркер початку аудіо. Це діє як жорстка умова (conditioning) для трансформера.
*   **Авторегресивний цикл:** Модель ітеративно передбачає логіти наступного токена.
*   **Стратегія семплювання:** Для уникнення деградації генерації (зациклення) використовується `Top-K` семплювання у комбінації з температурним масштабуванням (`temperature`). Це залишає лише $K$ найбільш імовірних варіантів, з яких токен обирається випадковим чином пропорційно до їхніх ймовірностей (функція `softmax` та `multinomial`).
*   **Зупинка:** Цикл переривається природним шляхом, якщо модель генерує токен `EOS_AUDIO`, або примусово, коли досягається ліміт `max_audio_tokens`. Наприкінці текстова частина промпту відсікається, і метод повертає виключно масив згенерованих акустичних кодів.

In [ ]:
class GPT2TTS(nn.Module):
    def __init__(self, vocab_size, audio_vocab_size, max_seq_length=1024):
        super().__init__()
        self.vocab_size = vocab_size
        self.audio_vocab_size = audio_vocab_size
        self.total_vocab_size = vocab_size + audio_vocab_size + 4 # +4 для спеціальних токенів

        # Спеціальні токени для розділення модальностей
        self.BOS_TEXT = self.total_vocab_size - 4
        self.EOS_TEXT = self.total_vocab_size - 3
        self.BOS_AUDIO = self.total_vocab_size - 2
        self.EOS_AUDIO = self.total_vocab_size - 1

        # Конфігурація GPT-2
        config = GPT2Config(
            vocab_size=self.total_vocab_size,
            n_positions=max_seq_length,
            n_embd=512,
            n_layer=6,   # Зменшено для швидкості експериментів (в оригіналі 12)
            n_head=8,
        )
        self.gpt = GPT2LMHeadModel(config)

    def forward(self, input_ids, labels=None):
        # GPT-2 автоматично рахує CrossEntropyLoss, якщо передати labels
        return self.gpt(input_ids=input_ids, labels=labels)

    @torch.no_grad()
    def generate(self, text_tokens, max_audio_tokens=500, temperature=1.0, top_k=50):
        """Функція для інференсу (генерації аудіо-токенів з тексту)"""
        self.gpt.eval()
        device = text_tokens.device

        # Формуємо промпт: [BOS_TEXT] ...text... [EOS_TEXT] [BOS_AUDIO]
        prompt = torch.cat([
            torch.tensor([self.BOS_TEXT], device=device),
            text_tokens,
            torch.tensor([self.EOS_TEXT, self.BOS_AUDIO], device=device)
        ]).unsqueeze(0) # Додаємо batch dimension

        generated = prompt

        for _ in range(max_audio_tokens):
            outputs = self.gpt(generated)
            next_token_logits = outputs.logits[:, -1, :] / temperature

            # Top-K семплінг
            indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
            next_token_logits[indices_to_remove] = -float('Inf')

            probs = torch.nn.functional.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            generated = torch.cat((generated, next_token), dim=1)

            if next_token.item() == self.EOS_AUDIO:
                break

        # Повертаємо тільки згенеровані аудіо-токени
        audio_start_idx = prompt.size(1)
        return generated[0, audio_start_idx:-1] # Відкидаємо EOS_AUDIO

У межах виконання лабораторної роботи лекційний шаблон класу `GPT2TTS` було перетворено на функціональну архітектуру. Для спільної обробки тексту та звуку ми реалізували об'єднаний словник, жорстко зафіксувавши чотири спеціальні маркери (`BOS/EOS` для обох модальностей), щоб модель чітко розпізнавала межу між текстовою умовою та цільовим аудіо. Крім того, конфігурацію базової моделі `GPT-2` було свідомо оптимізовано: кількість шарів зменшено до 6, а розмірність ембедингів — до 512. Це архітектурне спрощення дозволило адаптувати модель до лімітів відеопам'яті середовища Colab та суттєво пришвидшити експерименти без критичної втрати якості авторегресії.

Основна робота полягала в реалізації логіки навчання та інференсу. Для тренування було написано метод `forward`, який завдяки прямій передачі `labels` використовує вбудовані механізми маскування та автоматично рахує функцію втрат (Teacher Forcing). Найважливішим доопрацюванням стало створення методу `generate` з нуля. Замість базового шаблону ми реалізували повноцінний авторегресивний цикл, впровадивши `Top-K` семплювання з мультиноміальним вибором та умову ранньої зупинки. Це вирішило проблему генерації "білого шуму" і зациклення, дозволивши моделі синтезувати контрольовані та варіативні послідовності акустичних кодів.

# **PyTorch Lightning Module**

Клас `TTSLightningModule` є обгорткою на базі PyTorch Lightning, яка інкапсулює весь життєвий цикл підготовки даних та навчання мультимодальної моделі. Під час ініціалізації модуль об'єднує текстовий токенізатор (GPT-2), нейромережевий аудіокодек (Encodec) та кастомну архітектуру `GPT2TTS`. Ключовою складовою є метод `prepare_sequence`, який виконує складну попередню обробку батчу: він токенізує текст, здійснює безпечний ресемплінг аудіохвилі (на рівні CPU для уникнення апаратних конфліктів бібліотек) та квантує звук у дискретні токени. Для уникнення колізій індекси акустичних кодів математично зсуваються на розмір текстового словника, після чого обидві модальності склеюються в єдиний тензор зі структурними маркерами.

Другим надважливим компонентом є логіка тренування, реалізована в методі `training_step`. Оскільки мета моделі — генерувати звук на основі заданого тексту, а не прогнозувати сам текст, застосовано механізм цілеспрямованого маскування. Цільова послідовність (`labels`) створюється як копія вхідної, проте всім текстовим токенам (аж до маркера `BOS_AUDIO`) примусово встановлюється значення `-100`. Завдяки цьому внутрішня функція втрат трансформера повністю ігнорує текстову частину і розраховує градієнти (через оптимізатор `AdamW`) виключно для авторегресивного передбачення наступних акустичних токенів.

In [ ]:
class TTSLightningModule(pl.LightningModule):
    def __init__(self, text_tokenizer_name="gpt2", learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()

        self.tokenizer = AutoTokenizer.from_pretrained(text_tokenizer_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.audio_codec = EncodecModel.encodec_model_24khz()
        self.audio_codec.set_target_bandwidth(1.5)
        self.audio_vocab_size = self.audio_codec.quantizer.bins

        self.model = GPT2TTS(
            vocab_size=self.tokenizer.vocab_size,
            audio_vocab_size=self.audio_vocab_size
        )

    def forward(self, input_ids, labels=None):
        return self.model(input_ids, labels)

    def prepare_sequence(self, text_str, audio_waveform, sample_rate):
        device = self.device

        # Токенізація тексту
        text_tokens = self.tokenizer(text_str, return_tensors="pt").input_ids[0].to(device)

        # Робимо ресемплінг на cpu, щоб уникнути конфлікту torchaudio
        audio_waveform = convert_audio(
            audio_waveform.cpu(),
            sample_rate,
            self.audio_codec.sample_rate,
            self.audio_codec.channels
        )
        # 2. Переносимо відформатоване аудіо на відеокарту
        audio_waveform = audio_waveform.to(device).unsqueeze(0)

        with torch.no_grad():
            encoded_frames = self.audio_codec.encode(audio_waveform)
            audio_codes = encoded_frames[0][0]
            audio_tokens = audio_codes[0, 0, :]

        audio_tokens = audio_tokens + self.model.vocab_size

        input_ids = torch.cat([
            torch.tensor([self.model.BOS_TEXT], device=device),
            text_tokens,
            torch.tensor([self.model.EOS_TEXT, self.model.BOS_AUDIO], device=device),
            audio_tokens,
            torch.tensor([self.model.EOS_AUDIO], device=device)
        ])

        return input_ids

    def training_step(self, batch, batch_idx):
        texts, waveforms, sample_rates = batch
        loss = 0

        for text, waveform, sr in zip(texts, waveforms, sample_rates):
            input_ids = self.prepare_sequence(text, waveform, sr)

            labels = input_ids.clone()
            bos_audio_idx = (input_ids == self.model.BOS_AUDIO).nonzero(as_tuple=True)[0].item()
            labels[:bos_audio_idx + 1] = -100

            input_ids = input_ids.unsqueeze(0)
            labels = labels.unsqueeze(0)

            outputs = self(input_ids=input_ids, labels=labels)
            loss += outputs.loss

        loss = loss / len(texts)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.hparams.learning_rate)
        return optimizer

# **Створення DataModule для LJ Speech**

Клас `LJSpeechDataModule` реалізує надійний конвеєр підготовки даних, використовуючи `torchaudio` для безконфліктного завантаження датасету та впроваджуючи жорстку префільтрацію довгих записів для запобігання переповненню відеопам'яті (OOM). Крім того, модуль містить кастомний метод `collate_fn`, який динамічно розпаковує сирі кортежі та формує структуровані батчі (нормалізований текст, тензори аудіо, частота дискретизації) для правильної передачі в модель.

In [ ]:
class LJSpeechDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=2, num_workers=2):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        print("Завантаження датасету LJ Speech через torchaudio (надійно і без помилок)...")
        # Завантажуємо через нативний torchaudio.
        full_dataset = torchaudio.datasets.LJSPEECH(root="./", download=True)

        print("Фільтрація довгих записів для економії пам'яті (це займе близько хвилини)...")
        filtered_indices = []
        for i in range(len(full_dataset)):
            # full_dataset[i] повертає кортеж: (waveform, sample_rate, transcript, normalized_transcript)
            # normalized_transcript знаходиться під індексом 3
            if len(full_dataset[i][3]) < 150:
                filtered_indices.append(i)

        # Створюємо Subset лише з коротких аудіо
        filtered_dataset = Subset(full_dataset, filtered_indices)

        # Розбиваємо на тренувальну (95%) та валідаційну (5%) вибірки
        val_size = int(len(filtered_dataset) * 0.05)
        train_size = len(filtered_dataset) - val_size

        self.train_dataset, self.val_dataset = random_split(
            filtered_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
        )
        print(f"Дані готові! Тренувальних зразків: {train_size}, Валідаційних: {val_size}")

    def collate_fn(self, batch):
        """
        Формує батч із нативних тензорів torchaudio
        """
        texts = [item[3] for item in batch] # item[3] = normalized_transcript
        waveforms = [item[0].float() for item in batch] # item[0] = waveform (Тензор [1, time])
        sample_rates = [item[1] for item in batch] # item[1] = sample_rate
        return texts, waveforms, sample_rates

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=self.collate_fn,
            num_workers=self.num_workers,
            pin_memory=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=self.collate_fn,
            num_workers=self.num_workers,
            pin_memory=True
        )

# **Ініціалізація та запуск тренування**

У цьому блоці коду реалізовано повний цикл конфігурації, навчання та відновлення моделі за допомогою інфраструктури PyTorch Lightning. Ключовим елементом є налаштування механізму автозбереження (`ModelCheckpoint`), який інтегровано з хмарним сховищем для резервного копіювання ваг моделі кожні 5 епох, зберігаючи найостаннішу версію та топ-3 найкращих чекпоінтів на основі метрики помилки навчання. Об'єкт `Trainer` конфігурується на 50 епох із використанням апаратного прискорення та накопичення градієнтів (`accumulate_grad_batches=4`), що дозволяє ефективно симулювати більший розмір батчу в умовах обмеженої відеопам'яті.

Після успішного завершення тренувального циклу (`trainer.fit`) реалізовано логіку підготовки до інференсу. Код ідентифікує збережені файли та виконує завантаження конкретного "досвідченого" стану моделі за допомогою методу `load_from_checkpoint`. Наприкінці відновлена модель переводиться в детермінований режим оцінки (`eval()`) та завантажується на графічний прискорювач, що є необхідною умовою для коректної генерації аудіо без впливу шарів регуляризації.

In [ ]:
data_module = LJSpeechDataModule(batch_size=4)

In [ ]:
model = TTSLightningModule(learning_rate=5e-5)

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath='/content/drive/MyDrive/TTS_Model/', # Шлях на твоєму Google Диску
    filename='tts-epoch{epoch:02d}-loss{train_loss:.2f}',
    auto_insert_metric_name=False,
    save_top_k=3,           # Зберігаємо 3 найкращі моделі (з найменшим loss)
    monitor='train_loss',   # Орієнтуємося на графік помилки
    mode='min',
    every_n_epochs=5,       # Робити бекап кожні 5 епох
    save_last=True          # Завжди зберігати найостаннішу версію
)

In [ ]:
trainer = pl.Trainer(
    max_epochs=50,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
    accumulate_grad_batches=4,
    enable_checkpointing=True,
    callbacks=[checkpoint_callback] #
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
print("Починаємо тренування моделі...")
trainer.fit(model, datamodule=data_module)
print("Тренування завершено!")

Починаємо тренування моделі...
Завантаження датасету LJ Speech через torchaudio (надійно і без помилок)...
Фільтрація довгих записів для економії пам'яті (це займе близько хвилини)...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Дані готові! Тренувальних зразків: 11731, Валідаційних: 617


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type         ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ audio_codec │ EncodecModel │ 14.9 M │ eval │     0 │
│ 1 │ model       │ GPT2TTS      │ 45.7 M │ eval │     0 │
└───┴─────────────┴──────────────┴────────┴──────┴───────┘

Trainable params: 60.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 60.5 M                                                                                               
Total estimated model params size (MB): 242                                                                        
Modules in train mode: 0                                                                                           
Modules in eval mode: 413                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 414 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


In [ ]:
checkpoint_dir = '/content/drive/MyDrive/TTS_Model/'
print("Знайдені чекпоінти:")
for file in os.listdir(checkpoint_dir):
    if file.endswith(".ckpt"):
        print(file)

Знайдені чекпоінти:
tts-epoch04-loss3.34.ckpt
tts-epoch09-loss3.15.ckpt
last.ckpt


In [ ]:
checkpoint_path = '/content/drive/MyDrive/TTS_Model/tts-epoch09-loss3.15.ckpt'

print(f"Завантаження моделі з: {checkpoint_path}")

loaded_model = TTSLightningModule.load_from_checkpoint(
    checkpoint_path,
    text_tokenizer_name="gpt2",
    learning_rate=5e-5
)

# Переводимо в режим оцінки (вимикаємо навчання) і кидаємо на відеокарту
loaded_model.eval()
loaded_model.to("cuda" if torch.cuda.is_available() else "cpu")

Завантаження моделі з: /content/drive/MyDrive/TTS_Model/tts-epoch09-loss3.15.ckpt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


100%|██████████| 88.9M/88.9M [00:03<00:00, 24.1MB/s]


TTSLightningModule(
  (audio_codec): EncodecModel(
    (encoder): SEANetEncoder(
      (model): Sequential(
        (0): SConv1d(
          (conv): NormConv1d(
            (conv): Conv1d(1, 32, kernel_size=(7,), stride=(1,))
            (norm): Identity()
          )
        )
        (1): SEANetResnetBlock(
          (block): Sequential(
            (0): ELU(alpha=1.0)
            (1): SConv1d(
              (conv): NormConv1d(
                (conv): Conv1d(32, 16, kernel_size=(3,), stride=(1,))
                (norm): Identity()
              )
            )
            (2): ELU(alpha=1.0)
            (3): SConv1d(
              (conv): NormConv1d(
                (conv): Conv1d(16, 32, kernel_size=(1,), stride=(1,))
                (norm): Identity()
              )
            )
          )
          (shortcut): SConv1d(
            (conv): NormConv1d(
              (conv): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
              (norm): Identity()
            )
          )
   

# **Інференс та Експерименти з Семплінгом**

## **Розширена функція генерації**

Функція `advanced_generate` реалізує комплексний алгоритм авторегресивного інференсу для синтезу акустичних токенів на основі заданого тексту. На першому етапі функція токенізує вхідний рядок та конкатенує його зі спеціальними структурними маркерами, формуючи початковий промпт, який слугує жорсткою умовою для генерації. У циклі передбачення розраховуються логіти наступного токена, до яких застосовується температурне масштабування та одна з обраних стратегій семплювання: детермінований вибір максимального значення (Greedy) або ймовірнісні методи обмеження розподілу (Top-K чи Top-P/Nucleus Sampling), що дозволяє балансувати між передбачуваністю та природною варіативністю звучання. Після завершення генерації (через досягнення ліміту або генерацію маркера зупинки) алгоритм ізолює цільові аудіо-токени, математично відновлює їхні оригінальні індекси та застосовує операцію обмеження (clamping) для уникнення апаратних помилок декодування у випадку можливих галюцинацій моделі.

In [ ]:
@torch.no_grad()
def advanced_generate(model_module, text_str, strategy="top_k", max_tokens=500, temp=1.0, k=50, p=0.9):
    device = model_module.device
    model_module.eval()

    # 1. Токенізація вхідного тексту
    text_tokens = model_module.tokenizer(text_str, return_tensors="pt").input_ids[0].to(device)

    # 2. Формуємо стартовий промпт (як під час навчання)
    bos_text = torch.tensor([model_module.model.BOS_TEXT], device=device)
    eos_text = torch.tensor([model_module.model.EOS_TEXT], device=device)
    bos_audio = torch.tensor([model_module.model.BOS_AUDIO], device=device)
    eos_audio = model_module.model.EOS_AUDIO

    generated = torch.cat([bos_text, text_tokens, eos_text, bos_audio]).unsqueeze(0)

    # 3. Авторегресивна генерація аудіо-токенів
    for _ in range(max_tokens):
        outputs = model_module(input_ids=generated)
        # Беремо логіти лише останнього згенерованого токена і застосовуємо температуру
        logits = outputs.logits[:, -1, :] / temp

        if strategy == "greedy":
            # Просто беремо максимальне значення
            next_token = torch.argmax(logits, dim=-1, keepdim=True)

        else:
            if strategy == "top_k":
                # Обрізаємо все, що нижче K-того значення
                v, _ = torch.topk(logits, k)
                logits[logits < v[:, [-1]]] = -float('Inf')

            elif strategy == "top_p":
                # Nucleus sampling
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

                # Видаляємо токени, чия кумулятивна ймовірність перевищує p
                sorted_indices_to_remove = cumulative_probs > p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                logits[indices_to_remove] = -float('Inf')

            # Перетворюємо логіти у ймовірності і робимо зважений випадковий вибір
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        generated = torch.cat((generated, next_token), dim=1)

        # Зупиняємося, якщо модель вирішила, що аудіо закінчилося
        if next_token.item() == eos_audio:
            break

    # 4. Витягуємо та очищаємо аудіо-токени
    audio_start_idx = len(text_tokens) + 3 # пропускаємо текстові токени та спец-токени
    audio_tokens = generated[0, audio_start_idx:-1] # відкидаємо фінальний eos_audio

    # Зсуваємо індекси назад (бо під час тренування ми додавали vocab_size)
    audio_tokens = audio_tokens - model_module.model.vocab_size
    # Захист: якщо модель згалюцинувала текстовий токен, примусово стискаємо його до діапазону аудіо
    audio_tokens = torch.clamp(audio_tokens, 0, model_module.audio_vocab_size - 1)

    return audio_tokens

## **Функція декодування**

Функція `decode_audio` форматує згенеровану послідовність акустичних токенів у необхідну тензорну структуру та виконує їхнє зворотне перетворення на кінцеву аудіохвилю за допомогою нейромережевого декодера Encodec.

In [ ]:
def decode_audio(model_module, audio_tokens):
    # Encodec очікує формат: [batch, K, time], де K - кількість кодових книг.
    # Ми тренували лише на K=1 (перший шар для простоти)
    codes = audio_tokens.unsqueeze(0).unsqueeze(0).to(model_module.device)

    with torch.no_grad():
        # Encodec API вимагає передавати дані списком кортежів
        encoded_frames = [(codes, None)]
        waveform = model_module.audio_codec.decode(encoded_frames)

    return waveform[0].cpu() # Повертаємо тензор хвилі

## **Запуск генерації**

Цей блок коду виконує практичний інференс моделі для тестової фрази, послідовно застосовуючи три різні стратегії семплювання (greedy, top-k, top-p) для синтезу, декодування, збереження та безпосереднього відтворення отриманих аудіофайлів.

In [ ]:
# Текст для перевірки
test_text = "The quick brown fox jumps over the lazy dog."
strategies = ["greedy", "top_k", "top_p"]

print(f"Текст для генерації: '{test_text}'")
print("-" * 50)

# Переводимо модель на GPU (на всякий випадок)
loaded_model.to("cuda" if torch.cuda.is_available() else "cpu")

for strat in strategies:
    print(f"Семплінг: {strat.upper()} ... генерація")

    # Генеруємо токени (для top_k беремо 50, для top_p беремо 0.9)
    tokens = advanced_generate(loaded_model, test_text, strategy=strat, temp=1.0, k=50, p=0.9)

    # Перетворюємо на звук
    wav = decode_audio(loaded_model, tokens)

    # Зберігаємо файл (він знадобиться для розрахунку метрик на наступному етапі)
    filename = f"generated_{strat}.wav"
    torchaudio.save(filename, wav, sample_rate=24000) # Encodec працює на 24kHz

    # Виводимо плеєр
    display(Audio(filename))
    print("-" * 50)

Текст для генерації: 'The quick brown fox jumps over the lazy dog.'
--------------------------------------------------
Семплінг: GREEDY ... генерація


Audio(sampling_rate='generated_greedy.wav', decode=True, stream_index=None)

--------------------------------------------------
Семплінг: TOP_K ... генерація


Audio(sampling_rate='generated_top_k.wav', decode=True, stream_index=None)

--------------------------------------------------
Семплінг: TOP_P ... генерація


Audio(sampling_rate='generated_top_p.wav', decode=True, stream_index=None)

--------------------------------------------------


In [ ]:
# Список стратегій, які ми використовували
strategies = ["greedy", "top_k", "top_p"]

print("=== Прослуховування згенерованих аудіо ===")
#  "The quick brown fox jumps over the lazy dog."

for strat in strategies:
    filename = f"generated_{strat}.wav"

    # Перевіряємо, чи файл дійсно зберігся
    if os.path.exists(filename):
        print(f"\nМетод семплінгу: {strat.upper()}")
        display(Audio(filename))
    else:
        print(f"\nФайл {filename} не знайдено. Переконайся, що генерація пройшла успішно.")

=== Прослуховування згенерованих аудіо ===

Метод семплінгу: GREEDY



Метод семплінгу: TOP_K



Метод семплінгу: TOP_P


# **Оцінка**

## **Ініціалізація моделей-оцінювачів**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Розрахунок на: {device.upper()}...")

Розрахунок на: CPU...


In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-base")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base").to(device)
cer_metric = evaluate.load("cer")

print("Завантаження моделі SpeechBrain (для SECS)...")
spk_classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": device}
)

print("Завантаження моделі UTMOS (для оцінки природності)...")
utmos_predictor = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True)
utmos_predictor = utmos_predictor.to(device)
utmos_predictor.eval()

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


Завантаження моделі SpeechBrain (для SECS)...


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Завантаження моделі UTMOS (для оцінки природності)...


Using cache found in /root/.cache/torch/hub/tarepan_SpeechMOS_v1.2.0
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


UTMOS22Strong(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): ConvFeatureExtractionModel(
      (conv_layers): ModuleList(
        (0): Sequential(
          (0): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (1): Dropout(p=0.0, inplace=False)
          (2): GroupNorm(512, 512, eps=1e-05, affine=True)
          (3): GELU(approximate='none')
        )
        (1-4): 4 x Sequential(
          (0): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (1): Dropout(p=0.0, inplace=False)
          (2): GELU(approximate='none')
        )
        (5-6): 2 x Sequential(
          (0): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (1): Dropout(p=0.0, inplace=False)
          (2): GELU(approximate='none')
        )
      )
    )
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (post_extract_proj): Linear(in_features=512, out_features=768, bias=True)
    (dropout_input): Dropout(p=0.1, inplace=False)


## **Підготовка референсного аудіо**

In [ ]:
# Примусово ініціалізуємо і готуємо дані, якщо вони зникли з пам'яті
data_module = LJSpeechDataModule(batch_size=1)
data_module.setup()

# Тепер train_dataset точно існує! Беремо перший-ліпший файл
ref_waveform = data_module.train_dataset[0][0] # Оригінальний тензор
ref_sample_rate = data_module.train_dataset[0][1]

# SpeechBrain очікує 16000 Hz
if ref_sample_rate != 16000:
    ref_waveform = torchaudio.functional.resample(ref_waveform, ref_sample_rate, 16000)

with torch.no_grad():
    ref_embedding = spk_classifier.encode_batch(ref_waveform.to(device))

Підготовка референсного аудіо...
Завантаження датасету LJ Speech через torchaudio (надійно і без помилок)...


100%|██████████| 2.56G/2.56G [00:51<00:00, 53.3MB/s]


Фільтрація довгих записів для економії пам'яті (це займе близько хвилини)...
Дані готові! Тренувальних зразків: 11731, Валідаційних: 617


## **Розрахунок метрик для кожної стратегії**

In [ ]:
strategies = ["greedy", "top_k", "top_p"]
original_text = "The quick brown fox jumps over the lazy dog."

results = []
print("\nПочинаємо розрахунок...\n")

for strat in strategies:
    # Перевіряємо обидва варіанти назв (з 1-ї епохи та з 9-ї)
    filename = f"generated_{strat}.wav"
    if not os.path.exists(filename):
        filename = f"epoch9_generated_{strat}.wav"
        if not os.path.exists(filename):
            continue

    print(f"Аналіз файлу: {filename}")

    gen_wav, sr = torchaudio.load(filename)
    gen_wav_16k = torchaudio.functional.resample(gen_wav, sr, 16000).to(device)

    with torch.no_grad():
        # --- РОЗРАХУНОК CER (НАДІЙНИЙ РУЧНИЙ МЕТОД) ---
        audio_array = gen_wav_16k[0].cpu().numpy()

        # 1. Екстракція ознак напряму
        input_features = processor(audio_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)
        # 2. Генерація токенів
        predicted_ids = whisper_model.generate(input_features)
        # 3. Декодування в текст
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        pred_text = transcription.lower().strip()
        ref_text = original_text.lower().strip()
        cer_score = cer_metric.compute(predictions=[pred_text], references=[ref_text])

        # --- РОЗРАХУНОК SECS ---
        gen_embedding = spk_classifier.encode_batch(gen_wav_16k)
        secs_score = torch.nn.functional.cosine_similarity(ref_embedding, gen_embedding, dim=-1).mean().item()

        # --- РОЗРАХУНОК UTMOS ---
        utmos_score = utmos_predictor(gen_wav_16k, 16000).item()

    results.append({
        "Семплінг": strat.upper(),
        "CER (менше=краще)": round(cer_score, 4),
        "UTMOS (більше=краще)": round(utmos_score, 2),
        "SECS (більше=краще)": round(secs_score, 4),
        "Розпізнаний текст": pred_text
    })

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.



Починаємо розрахунок...

Аналіз файлу: generated_greedy.wav


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Аналіз файлу: generated_top_k.wav
Аналіз файлу: generated_top_p.wav


In [ ]:
df_results = pd.DataFrame(results)
print("\n" + "="*70)
print("ФІНАЛЬНІ МЕТРИКИ:")
print("="*70)
display(df_results)


ФІНАЛЬНІ МЕТРИКИ (Для звіту):


,Семплінг,CER (менше=краще),UTMOS (більше=краще),SECS (більше=краще),Розпізнаний текст
0,GREEDY,0.8409,1.40,-0.0216,the plan!
1,TOP_K,0.9545,1.26,0.0480,the trump public insist on a decimation to our...
2,TOP_P,1.0682,1.28,0.1997,"the parts were the verses of cells, dealing wi..."
